# 🎭 AI Avatar Video Generator

**Run each cell in order (▶ button or Shift+Enter).**

| Cell | What it does |
|------|-------------|
| 0 | ⚠️ Set runtime to **2025.07** (required for voice cloning) |
| 1 | Mount Google Drive |
| 2 | Clone repo + install dependencies |
| 3 | Download model weights |
| 4 | Verify GPU |
| 5 | Launch the Gradio app |

> **Important — do this BEFORE running any cells:**
> 1. **Runtime → Change runtime type**
> 2. **Hardware accelerator:** GPU → **T4**
> 3. **Runtime version:** **2025.07** (Python 3.11 — MeloTTS does not work on 3.12)
> 4. Click **Save**, then **Runtime → Restart session**

## Cell 0 — Check Python version (must be 3.11)

Run this first. If you see Python **3.12**, go back and set **Runtime version → 2025.07**, then restart.

In [ ]:
import sys

v = sys.version_info
print(f"Python {v.major}.{v.minor}.{v.micro}")

if v >= (3, 12):
    raise RuntimeError(
        "Python 3.12 detected — MeloTTS (voice cloning) will NOT install.\n\n"
        "Fix:\n"
        "  Runtime → Change runtime type → Runtime version → 2025.07\n"
        "  Save → Runtime → Restart session → re-run this cell"
    )
else:
    print("✅ Python version OK — continue to Cell 1")

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/AvatarOutput', exist_ok=True)
print("✅ Drive mounted. Output will be saved to: /content/drive/MyDrive/AvatarOutput")

## Cell 2 — Clone repo and run setup

> Replace `YOUR_USERNAME` with your GitHub username if you have forked the repo.

In [ ]:
import os

REPO_URL = "https://github.com/YOUR_USERNAME/avatar_pipeline.git"
REPO_DIR = "/content/avatar_pipeline"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repo already cloned at {REPO_DIR}")

os.chdir(REPO_DIR)
!bash setup/install.sh

## Cell 3 — Download model weights

> This downloads SadTalker, OpenVoice v2, and MusePose checkpoints (~8 GB total).
> Skip files that are already present — safe to re-run.

In [ ]:
!python setup/download_weights.py

## Cell 4 — Verify GPU and VRAM

In [ ]:
import torch

cuda_ok = torch.cuda.is_available()
print(f"CUDA available : {cuda_ok}")

if cuda_ok:
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    print(f"GPU            : {props.name}")
    print(f"VRAM           : {vram_gb:.1f} GB")
    print(f"CUDA version   : {torch.version.cuda}")
    if vram_gb < 14:
        print("\n⚠️  Less than 14 GB VRAM detected. Use 256 resolution and avoid loading "
              "SadTalker and OpenVoice models at the same time.")
    else:
        print("\n✅ VRAM looks good — you can use 512 resolution if desired.")
else:
    print("\n❌ No GPU detected. Go to Runtime → Change runtime type → GPU → T4.")

## Cell 5 — Launch the Gradio app

> After running this cell you will see a public URL like
> `https://abc123.gradio.live` — click it to open the UI.
>
> The app runs until you interrupt this cell (■ Stop button).

In [ ]:
import os
os.chdir('/content/avatar_pipeline')
!python app.py